---

### Python Workshop Part 13: Goal Seek

© Kerry Back, Rice University

---

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, fsolve
from scipy.linalg import solve
import warnings
warnings.filterwarnings('ignore')

### scipy.optimize.fsolve 

Excel's Goal Seek changes one input cell to achieve a desired value in a formula cell. Python's `fsolve` does that, but it can also solve for multiple input values simultaneously.

**Python fsolve:**
- Define an equation that equals zero at your target
- fsolve finds the input(s) that makes it zero


`fsolve` returns a numpy array.  Recall that the elements of that array are extracted as `arr[0]`, `arr[1]`, etc.

Here is a simple example of finding a single input value to achieve a desired value.  We know from algebra (factoring or quadratic formula) that there are two solutions x = 2 and x = 3.

In [ ]:

# Question: What value of x makes x² - 5x + 6 = 0?
print("="*60)
print("fsolve EXAMPLE 1")
print("="*60)

# STEP 1: Define a function that is zero at the value we want to find
def function_to_find_zero(x):
    return x**2 - 5*x + 6
   
# STEP 2: Find solution(s).  Vary initial values to look for multiple solutions.
initial_value1 = 0
solution1 = fsolve(function_to_find_zero, initial_value1)
initial_value2 = 5
solution2 = fsolve(function_to_find_zero, initial_value2)

print(f"\nfsolve Results:")
print(f"  Starting near 0: solution = {solution1}")
print(f"  Starting near 5: solution = {solution2}")

# STEP 3: Verify answers
print("\nVERIFYING ANSWERS")
print(f"2**2 -5*2 + 6 = {function_to_find_zero(2)}")
print(f"3**2 -5*3 + 6 = {function_to_find_zero(3)}")

### Exercise 1

**Problem:** Find the values of x where x³ - 6x² = 6 - 11x.  Verify your answers.

**Hints:**
- This equation has three solutions.
- Try different starting guesses to find all three solutions.
- Remember that you need to define a function that is zero at the answer,

### `fsolve` Example 2

The following presents a more complicated example, similar to something we might solve in Excel.  

`fsolve` can also handle even more complicated examples, including solving for multiple variables simultaneously.

In [ ]:
# Question: What annual sales volume gives NPV = 0?
print("="*60)
print("fsolve EXAMPLE 2")
print("="*60)

# Project assumptions
initial_capex = 100000      # Initial capital expenditure
project_life = 5            # Years
discount_rate = 0.10        # 10% discount rate

# Operating assumptions
unit_price = 50             # Selling price per unit
cogs_per_unit = 25          # Cost of goods sold per unit
sga_annual = 5000           # Selling, general & administrative costs
tax_rate = 0.30             # 30% tax rate
nwc_per_unit = 9            # Net working capital per unit

# Depreciation (straight-line)
annual_depreciation = initial_capex / project_life

print("Project Assumptions:")
print(f"  Initial CapEx: ${initial_capex:,}")
print(f"  Project Life: {project_life} years")
print(f"  Discount Rate: {discount_rate:.1%}")
print(f"  Unit Price: ${unit_price}")
print(f"  COGS per Unit: ${cogs_per_unit}")
print(f"  Annual SG&A: ${sga_annual:,}")
print(f"  Tax Rate: {tax_rate:.1%}")
print(f"  NWC: ${nwc_per_unit} per unit")
print(f"  Depreciation: ${annual_depreciation:,} per year (straight-line)")

  
def calculate_npv(annual_qty):
    """Calculate NPV for a given annual sales quantity"""
    
    # Initialize cash flows array
    cash_flows = [-initial_capex]  # Year 0: Initial investment
    
    # Calculate working capital schedule
    nwc_balances = []
    
    for year in range(1, project_life + 1):
        # Annual financial calculations
        sales = annual_qty * unit_price
        cogs = annual_qty * cogs_per_unit
        gross_profit = sales - cogs
        
        # EBITDA = Gross Profit - SG&A
        ebitda = gross_profit - sga_annual
        
        # EBIT = EBITDA - Depreciation
        ebit = ebitda - annual_depreciation
        
        # Net Income = EBIT × (1 - Tax Rate)
        net_income = ebit * (1 - tax_rate)
        
        # Net Working Capital
        nwc = annual_qty * nwc_per_unit
        nwc_balances.append(nwc)
        
        # Change in NWC (cash outflow if positive)
        if year == 1:
            change_nwc = nwc  # Initial NWC investment
        else:
            change_nwc = nwc - nwc_balances[year-1]  # Change from previous year
        
        # Cash Flow = Net Income + Depreciation - Change in NWC
        if year < project_life:
            cash_flow = net_income + annual_depreciation - change_nwc
        else:
            # Final year: recover all NWC
            cash_flow = net_income + annual_depreciation + nwc  # Add back all NWC
        
        cash_flows.append(cash_flow)
        
    # Calculate NPV
    npv = 0
    for year, cf in enumerate(cash_flows):
        npv += cf / (1 + discount_rate)**year
    
    return npv

# Demonstrate NPV calculation with example quantity
example_qty = 5000
example_npv = calculate_npv(example_qty)
print(f"\nExample: At {example_qty:,} units/year, NPV = ${example_npv:,.0f}")


# We want to set NPV = 0, so we can use calculate_npv as function_to_find_zero
initial_guess = 4000
breakeven_qty = fsolve(calculate_npv, initial_guess)[0]
print(f"\nRequired Annual Sales Volume: {breakeven_qty:,.0f} units")

# Verify NPV = 0
verification_npv = calculate_npv(breakeven_qty)
print(f"\nVerification:")
print(f"  NPV at break-even volume: ${verification_npv:,.2f} ✓")

## scipy.optimize.minimize

While `fsolve` finds where functions equal zero, `minimize` finds where functions reach their lowest value.

The next cell presents a simple example of using `minimize` to find the value of $x$ that minimizes 
$$x^2 - 4x+7$$
This can easily be found by calculus, but we will use the `minimize` function instead.

The `minimize` function returns an object, which is named `result` in the following example.  The most important attributes of the object are:

- `result.success`: was the minimization successful (True or False)
- `result.x`: the value of $x$ that minimizes the function
- `result.fun`: the value of the function at the minimum


In [ ]:
# Question: What value of x minimizes f(x) = x² - 4x + 7?
print("="*60)
print("minimize EXAMPLE 1")
print("="*60)

def function_to_minimize(x):
        return x**2 - 4*x + 7

initial_guess = 0
result = minimize(function_to_minimize, initial_guess)
print(f'The result object is \n{result}\n')
print(f'Was the minimization successful? {result.success}')
print(f'What is the minimizing value of x? {result.x}')
print(f'What is the value of the function at the minimum? {result.fun}')



### If We Want to Maximize

The minimize function can solve both minimize and maximize problems.  To solve a maximize problem, we minimize the negative of the function.  This is illustrated in the figure below.

You are not expected to read the code in the following cell, but you are welcome to if time permits.

In [ ]:
# Visualization: Maximization via Minimization
# This figure illustrates why we minimize the negative to maximize a function

import numpy as np
import matplotlib.pyplot as plt

# Create x values
x = np.linspace(-3, 7, 200)

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# LEFT SUBPLOT: 
fun_to_max1 = - (x - 2)**2 - 3
fun_to_min1 = -fun_to_max1

ax1.plot(x, fun_to_max1, 'r-', linewidth=2, label='$f(x) = -(x-2)^2 - 3$')
ax1.plot(x, fun_to_min1, 'b-', linewidth=2, label='$-f(x) = (x-2)^2 + 3$')

# Annotate minimum of -f(x)
ax1.plot(2, 3, 'bo', markersize=10)
ax1.annotate('Minimum of $-f(x)$',
             xy=(2, 3), xytext=(3.5, 5),
             arrowprops=dict(arrowstyle='->', color='blue', lw=1.5),
             fontsize=10, color='blue', fontweight='bold')

# Annotate maximum of f(x)
ax1.plot(2, -3, 'ro', markersize=10)
ax1.annotate('Maximum of $f(x)$', 
             xy=(2, -3), xytext=(3.5, -5),
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
             fontsize=10, color='red', fontweight='bold')

ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.axvline(x=2, color='gray', linestyle='--', alpha=0.5)
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('Example 1', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right')
ax1.set_ylim(-8, 12)

# RIGHT SUBPLOT: 
fun_to_max2 = - (x - 3)**2 + 4
fun_to_min2 = -fun_to_max2

ax2.plot(x, fun_to_max2, 'r-', linewidth=2, label='$f(x) = -(x-3)^2 + 4$')
ax2.plot(x, fun_to_min2, 'b-', linewidth=2, label='$-f(x) = (x-3)^2 - 4$')

# Annotate minimum of -f(x)
ax2.plot(3, -4, 'bo', markersize=10)
ax2.annotate('Minimum of $-f(x)$', 
             xy=(3, -4), xytext=(4.5, -6),
             arrowprops=dict(arrowstyle='->', color='blue', lw=1.5),
             fontsize=10, color='blue', fontweight='bold')

# Annotate maximum of f(x)
ax2.plot(3, 4, 'ro', markersize=10)
ax2.annotate('Maximum of $f(x)$', 
             xy=(3, 4), xytext=(4.5, 6),
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
             fontsize=10, color='red', fontweight='bold')

ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(x=3, color='gray', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('y', fontsize=12)
ax2.set_title('Example 2', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.set_ylim(-8, 12)

# Overall title
fig.suptitle('Key Concept: To MAXIMIZE $f(x)$, we MINIMIZE $-f(x)$', fontsize=16, fontweight='bold', y=1.02)

# Add text box with explanation
textstr = ('Notice: The $x$ value that minimizes $-f(x)$ is the SAME $x$ value that maximizes $f(x)$\n'
           'This is why we minimize the negative when we want to maximize!')
fig.text(0.5, -0.05, textstr, ha='center', fontsize=11, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

### Exercise 2

**Scenario:** A monopolist wants to choose the optimal output level to maximize profit.

**Problem Setup:**
- **Demand curve:** P = 100 - 2Q (price decreases as quantity increases)
- **Variable cost:** $20 per unit (constant)
- **Fixed cost:** $500
- **Revenue:** R(Q) = P × Q = (100 - 2Q) × Q = 100Q - 2Q²
- **Total Cost:** TC(Q) = 500 + 20Q
- **Profit:** π(Q) = R(Q) - TC(Q) = 100Q - 2Q² - 500 - 20Q = 80Q - 2Q² - 500
- **Goal:** Find the quantity Q that maximizes profit

### `minimize` Example 2

`minimize` can be used

- for multiple variables
- when the variables are bounded
- when there additional constraints

We need to specify:

- Objective function: Function to minimize
- Variables: Array of decision variables
- Constraints: Dictionary of constraint functions

The following example illustrates maximizing profit subject to resource constraints.

In [ ]:
print("="*60)
print("minimize Example 2")
print("="*60)

# Product data
products = ['Product A', 'Product B', 'Product C']
profits = np.array([20, 30, 25])  # Profit per unit

# Resource requirements per unit
labor_required = np.array([2, 3, 2.5])  # Hours per unit
material_required = np.array([3, 2, 4])  # Units per unit

# Available resources
labor_available = 500      # Total labor hours
material_available = 600    # Total material units

# Maximum demand
max_demand = np.array([100, 120, 80])

print("\nProducts:")
for i, product in enumerate(products):
    print(f"  {product}: ${profits[i]} profit, {labor_required[i]}h labor, "
            f"{material_required[i]} materials, max {max_demand[i]} units")

print(f"\nResources Available:")
print(f"  Labor: {labor_available} hours")
print(f"  Materials: {material_available} units")

# Define function to minimize
def function_to_min(quantities):
    """Negative profit (we minimize, so negate for maximization)"""
    return -np.dot(quantities, profits)  # np.dot is SUMPROD in Excel
    
# Define constraints
def labor_constraint(quantities):
    """Labor used must be <= available"""
    return labor_available - np.dot(quantities, labor_required)
    
def material_constraint(quantities):
    """Materials used must be <= available"""
    return material_available - np.dot(quantities, material_required)
    
# Dictionary of constraints
constraints = [
    {'type': 'ineq', 'fun': labor_constraint},    # >= 0
    {'type': 'ineq', 'fun': material_constraint}  # >= 0
]
    
# List of bounds
bounds = [(0, max_demand[i]) for i in range(len(products))]
    
# Initial guess
initial_quantities = np.array([50, 50, 40])
    
# Solve 
result = minimize(
    function_to_min, 
    initial_quantities, 
    method='SLSQP',
    bounds=bounds, 
    constraints=constraints
)

# Report result
success = result.success
optimal_quantities = result.x
optimal_profit = -result.fun
    
print(f"\nWas the optimization successful? {success}")
print(f"The optimal quantities are: {optimal_quantities}")
print(f"The maximum profit is: {optimal_profit}")
        


### Exercise 3: Marketing Budget Allocation

**Scenario:** You have a $100,000 marketing budget to allocate across three channels. Each channel has different returns and constraints.

**Channel Data:**
- Digital: $3 return per $1 spent, minimum $10,000, maximum $60,000
- TV: $2.5 return per $1 spent, minimum $20,000, maximum $50,000
- Print: $2 return per $1 spent, minimum $5,000, maximum $30,000

**Goal:** Maximize total return while spending no more than $100,000

You want to choose how much to spend on each channel.  Call these amounts digital_dollars, tv_dollars, and print_dollars.

- **Step 1.**  Define the function to minimize as function_to_min
- **Step 2.**  The maximum spending amounts for the channels are bounds on the choice variables.  Define a list of bounds.
- **Step 3.**  The limit on the total amount to spend is a constraint.  Define a dictionary of constraints.
- **Step 4.**  Define an intial_guess.
- **Step 5.**  Run `minimize` in the same way as in the previous example.
- **Step 6.**  Report the results in the same way as in the previous example.

Ask Gemini for help as needed or ask Gemini to check your work.